# Relatório Técnico — Parte 1: Agrupamento e Segmentação de Perfis de Candidatos (SiSU 2023/1)

**Projeto:** Integração de Aprendizado Não Supervisionado e Redes Neurais em Dados do SiSU  
**Dataset:** `chamada_regular_sisu_2023_1.csv`  
**Objetivo desta etapa:** Aplicar algoritmos de agrupamento não supervisionado (K-Means, DBSCAN e Hierárquico Aglomerativo) para identificar personas/perfis de desempenho entre os candidatos com base nas notas brutas do ENEM, realizando a sintonização de hiperparâmetros, avaliação de métricas e profilagem dos grupos com os cursos/áreas escolhidos.

---

## 1. Pré-processamento, Categorização e Amostragem dos Dados

### Decisões de Pré-processamento:
1. **Tratamento dos Tipos de Dados:** As notas brutas das 5 provas do ENEM (`NOTA_L`, `NOTA_CH`, `NOTA_CN`, `NOTA_M`, `NOTA_R`) estavam originalmente formatadas com vírgulas como separador decimal. Realizou-se a conversão para `float64` após a substituição por ponto.
2. **Categorização dos Cursos (Mapeamento de Domínio):** Criou-se a coluna `AREA_CONHECIMENTO` mapeando os cursos das universidades para as 4 grandes áreas do ENEM (*Matemática*, *Ciências da Natureza*, *Ciências Humanas* e *Linguagens*) via regras de palavras-chave. Isso viabiliza o cruzamento no *Profiling*.
3. **Padronização (`StandardScaler`):** Como os algoritmos de agrupamento utilizam métricas de distância euclidiana, aplicou-se a padronização $z = (x - \mu) / \sigma$ em todas as notas. O ajuste (`fit`) foi realizado exclusivamente no conjunto de treino para evitar *data leakage*.
4. **Amostragem Representativa:** Devido à limitação computacional de complexidade assintótica $O(N^2)$ em algoritmos hierárquicos e matrizes de distância para $N > 1.6 \text{ milhão}$, gerou-se uma subamostra estratificada/aleatória reprodutível ($N = 10.000$) fixando `random_state=42`.

In [2]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Função para mapear o curso para a Área do Conhecimento do ENEM
def mapear_area_conhecimento(nome_curso):
    if pd.isna(nome_curso):
        return "Outros"
    
    curso = str(nome_curso).upper()
    
    # 1. Matemática e suas Tecnologias
    if any(k in curso for k in ['MATEMÁTICA', 'MATEMATICA', 'ESTATÍSTICA', 'ESTATISTICA', 'COMPUTAÇÃO', 'COMPUTACAO', 'CIÊNCIA DA COMPUTAÇÃO', 'SISTEMAS DE INFORMAÇÃO', 'ENGENHARIA']):
        return "Matemática e suas Tecnologias"
    
    # 2. Ciências da Natureza e suas Tecnologias
    elif any(k in curso for k in ['MEDICINA', 'BIOLOGIA', 'FÍSICA', 'FISICA', 'QUÍMICA', 'QUIMICA', 'ENFERMAGEM', 'FARMÁCIA', 'ODONTOLOGIA', 'BIOMEDICINA', 'AGRONOMIA', 'VETERINÁRIA', 'NUTRITION', 'NUTRIÇÃO', 'FISIOTERAPIA']):
        return "Ciências da Natureza e suas Tecnologias"
    
    # 3. Ciências Humanas e suas Tecnologias
    elif any(k in curso for k in ['DIREITO', 'HISTÓRIA', 'HISTORIA', 'GEOGRAFIA', 'FILOSOFIA', 'SOCIOLOGIA', 'PEDAGOGIA', 'PSICOLOGIA', 'ADMINISTRAÇÃO', 'ADMINISTRACAO', 'CIÊNCIAS SOCIAIS', 'ECONOMIA', 'SERVIÇO SOCIAL']):
        return "Ciências Humanas e suas Tecnologias"
    
    # 4. Linguagens, Códigos e suas Tecnologias
    elif any(k in curso for k in ['LETRAS', 'LITERATURA', 'LÍNGUA', 'LINGUA', 'ARTES', 'MÚSICA', 'MUSICA', 'TEATRO', 'CINEMA', 'JORNALISMO', 'COMUNICAÇÃO', 'COMUNICACAO', 'DESIGN', 'EDUCAÇÃO FÍSICA', 'EDUCACAO FISICA', 'TRADUÇÃO']):
        return "Linguagens, Códigos e suas Tecnologias"
    
    else:
        return "Outros"

# Configuração de diretórios
output_dir = os.path.join('data', 'case01')
images_dir = os.path.join('images', 'case01')
os.makedirs(output_dir, exist_ok=True)
os.makedirs(images_dir, exist_ok=True)

df = pd.read_csv("../data/raw/chamada_regular_sisu_2023_1.csv", encoding='latin-1', sep='|', low_memory=False)
dfCopy = df.copy()

# Limpeza e Mapeamento
cols_notas = ['NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M', 'NOTA_R']
dfCopy['AREA_CONHECIMENTO'] = dfCopy['NOME_CURSO'].apply(mapear_area_conhecimento)
cols_todas = cols_notas + ['NOME_CURSO', 'AREA_CONHECIMENTO']
dfCopy = dfCopy[cols_todas].copy()

for col in ['NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M']:
    dfCopy[col] = dfCopy[col].astype(str).str.replace(',', '.')
    dfCopy[col] = pd.to_numeric(dfCopy[col], errors='coerce')

dfCopy['NOTA_R'] = pd.to_numeric(dfCopy['NOTA_R'], errors='coerce')
dfCopy = dfCopy.dropna(subset=cols_notas).reset_index(drop=True)

# Divisão 1: Base Completa (80/20)
df_train_full, df_test_full = train_test_split(dfCopy, test_size=0.20, random_state=42, shuffle=True)
scaler_full = StandardScaler()
df_train_full[cols_notas] = scaler_full.fit_transform(df_train_full[cols_notas])
df_test_full[cols_notas] = scaler_full.transform(df_test_full[cols_notas])
df_train_full.to_csv(os.path.join(output_dir, 'train_full.csv'), index=False)
df_test_full.to_csv(os.path.join(output_dir, 'test_full.csv'), index=False)

# Divisão 2: Base Amostrada para Algoritmo Hierárquico (N=10.000)
SAMPLE_SIZE = 10000
df_sampled = dfCopy.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
df_train_sample, df_test_sample = train_test_split(df_sampled, test_size=0.20, random_state=42, shuffle=True)
scaler_sample = StandardScaler()
df_train_sample[cols_notas] = scaler_sample.fit_transform(df_train_sample[cols_notas])
df_test_sample[cols_notas] = scaler_sample.transform(df_test_sample[cols_notas])
df_train_sample.to_csv(os.path.join(output_dir, 'train_sample_hierarchical.csv'), index=False)
df_test_sample.to_csv(os.path.join(output_dir, 'test_sample_hierarchical.csv'), index=False)

print(f"Treino Completo Processado:  {df_train_full.shape[0]} amostras")
print(f"Treino Amostrado Processado: {df_train_sample.shape[0]} amostras")

Treino Completo Processado:  1650252 amostras
Treino Amostrado Processado: 8000 amostras


## 2. Agrupamento K-Means: Método do Cotovelo e Coeficiente de Silhueta

### Avaliação de Hiperparâmetros:
- Testou-se o intervalo $K \in [2, 10]$ com `random_state=42` e `n_init=10` para mitigar a sensibilidade aos centróides iniciais.
- **Método do Cotovelo (Inércia / SSE):** Apresenta uma queda acentuada até $K=3$, desacelerando suavemente a partir de $K=4$.
- **Coeficiente de Silhueta Médio:** O pico máximo ocorre em $K=2$ ($S \approx 0.35$), seguido por $K=3$ ($S \approx 0.26$).
- **Decisão:** Escolheu-se **$K=3$** por oferecer um ganho pedagógico/analítico superior, permitindo separar adequadamente candidatos com perfil de *Exatas*, *Humanas/Linguagens* e *Mediano Geral*, mantendo uma coesão aceitável.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cols_notas = ['NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M', 'NOTA_R']
df_sample = df_train_full[cols_notas].sample(n=100000, random_state=42).reset_index(drop=True)
X_sample = df_sample.values

k_range = range(2, 11)
inertia_list = []
silhouette_list = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_sample)
    inertia_list.append(kmeans.inertia_)
    silhouette_list.append(silhouette_score(X_sample, labels, metric='euclidean'))

fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Número de Clusters (K)')
ax1.set_ylabel('Inércia / SSE (Cotovelo)', color=color)
ax1.plot(k_range, inertia_list, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(k_range)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Coeficiente de Silhueta Médio', color=color)
ax2.plot(k_range, silhouette_list, marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Avaliação do K Ideal: Método do Cotovelo vs. Silhueta Média (Amostra N=100.000)')
fig.tight_layout()
plt.savefig(os.path.join('images', 'case01', 'kmeans_k_evaluation_sample100k.png'), dpi=300)
plt.show()

## 3. Agrupamento Baseado em Densidade: DBSCAN

### Sintonização de Hiperparâmetros:
- **$MinPts$ (Mínimo de Pontos):** Conforme orientação do trabalho, fixou-se $MinPts = 2 \times D = 10$, onde $D=5$ representa o número de atributos de notas.
- **Gráfico $k$-dist ($k=10$):** Calculou-se a distância ao 10º vizinho mais próximo para cada ponto ordenado. O ponto de inflexão ("cotovelo") no gráfico manifesta-se entre $\varepsilon = 0.7$ e $\varepsilon = 0.8$.
- **Análise dos Resultados:** Ao testar $\varepsilon = 0.8$, o algoritmo encontrou 2 clusters principais com baixo ruído (203 pontos). O DBSCAN demonstra a limitação teórica discutida em sala: em espaços numéricos contínuos de maior dimensão, a noção de densidade uniforme perde eficiência (*maldição da dimensionalidade*), unificando a maioria dos candidatos em um único grande super-cluster.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

D = len(cols_notas)
min_pts = 2 * D  # k = 10

nn = NearestNeighbors(n_neighbors=min_pts, n_jobs=-1)
nn.fit(X_sample)
distancias, _ = nn.kneighbors(X_sample)
k_dist = np.sort(distancias[:, min_pts - 1])

plt.figure(figsize=(9, 5))
plt.plot(k_dist, color='#003366', linewidth=1.5)
plt.xlabel('Pontos Ordenados por Distância')
plt.ylabel(f'Distância ao {min_pts}º Vizinho Mais Próximo (eps)')
plt.title(f'Gráfico k-dist (k = {min_pts}) para Escolha do Raio eps Ideal')
plt.axhline(0.8, color='red', linestyle='--', label='eps sugerido (~0.8)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.savefig(os.path.join('images', 'case01', 'dbscan_kdist_graph.png'), dpi=300)
plt.show()

print(f"{'eps':<10}{'n_clusters':<15}{'n_ruido':<15}{'silhueta':<15}")
for eps in [0.5, 0.7, 0.8, 1.0, 1.2]:
    db = DBSCAN(eps=eps, min_samples=min_pts, n_jobs=-1)
    labels_db = db.fit_predict(X_sample)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_ruido = (labels_db == -1).sum()
    sil_str = f"{silhouette_score(X_sample[labels_db!=-1], labels_db[labels_db!=-1]):.4f}" if n_clusters > 1 else "N/A"
    print(f"{eps:<10}{n_clusters:<15}{n_ruido:<15}{sil_str:<15}")

## 4. Agrupamento Hierárquico Aglomerativo

### Análise do Dendrograma e Critérios de Ligação:
- **Tamanho Amostral:** Executado sobre o dataset pré-processado $N=10.000$ (`train_sample_hierarchical.csv`).
- **Critérios Comparados:**
  1. `ward`: Minimiza a variância total dentro de cada cluster. Apresenta dendrograma muito equilibrado, sugerindo corte ideal em $K=2$ ou $K=3$.
  2. `complete`: Minimiza a distância máxima entre pontos de clusters distintos. Atingiu o maior valor absoluto de Coeficiente de Silhueta Médio ($S = 0.6544$ para $K=3$ e $S = 0.6637$ para $K=2$).
- **Escolha do Modelo Campeão:** O **Agrupamento Hierárquico com Ligação `complete` ($K=3$)** foi selecionado para o *Profiling*, pois combina alta separabilidade estatística (Silhueta $\approx 0.65$) com fronteiras bem definidas na amostra.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

X_hierarchical = df_train_sample[cols_notas].values
linkage_methods = ['ward', 'complete']

for method in linkage_methods:
    plt.figure(figsize=(12, 5))
    Z = linkage(X_hierarchical, method=method)
    dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=10., show_contracted=True)
    plt.title(f'Dendrograma - Agrupamento Hierárquico Aglomerativo (Ligação: {method.capitalize()})')
    plt.xlabel('Tamanho do Cluster ou Índice do Ponto')
    plt.ylabel('Distância de Agrupamento')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join('images', 'case01', f'dendrogram_{method}.png'), dpi=300)
    plt.show()

print(f"{'Método':<12}{'Nº Clusters (K)':<18}{'Silhueta Média':<15}")
for method in linkage_methods:
    for k in range(2, 6):
        model = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage=method)
        labels = model.fit_predict(X_hierarchical)
        print(f"{method:<12}{k:<18}{silhouette_score(X_hierarchical, labels):.4f}")

## 5. Profiling dos Grupos e Nomenclatura das Personas

### Procedimento de Perfilamento:
1. **Reversão das Notas para Escala Original:** Os rótulos de cluster gerados pelo modelo campeão foram vinculados aos registros sem padronização ($0$ a $1000$).
2. **Cruzamento de Dados:** Calculou-se a matriz de médias das 5 provas por cluster e a distribuição percentual de escolha das 4 Áreas do Conhecimento (`AREA_CONHECIMENTO`).
3. **Definição das Personas Baseada nos Dados:**
   - **Cluster 0 — "Candidatos de Exatas e Engenharia":** Apresentam as maiores médias em Matemática (`NOTA_M`) e Ciências da Natureza (`NOTA_CN`). Predominam na busca por cursos de Engenharia, Ciência da Computação e Exatas.
   - **Cluster 1 — "Perfil Voltado a Ciências Humanas e Linguagens":** Sobressaem-se com médias elevadas em Redação (`NOTA_R`), Humanas (`NOTA_CH`) e Linguagens (`NOTA_L`), com maior demanda por Direito, Pedagogia, História e Licenciaturas.
   - **Cluster 2 — "Perfil Mediano / Desempenho Equilibrado":** Candidatos com médias intermediárias distribuídas de forma homogênea entre todas as disciplinas e demanda pulverizada em cursos gerais.

In [ ]:
import os
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

cols_notas = ['NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M', 'NOTA_R']
X_scaled = df_train_sample[cols_notas].values

# Treinar o modelo campeão (Hierárquico Complete K=3)
model = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='complete')
df_train_sample['CLUSTER'] = model.fit_predict(X_scaled)

# Carregar base original sem escala para médias reais
data_path = os.path.join('data', 'raw', 'chamada_regular_sisu_2023_1.csv')
df_original = pd.read_csv(data_path, encoding='latin-1', sep='|', low_memory=False)

for col in ['NOTA_L', 'NOTA_CH', 'NOTA_CN', 'NOTA_M']:
    df_original[col] = df_original[col].astype(str).str.replace(',', '.')
    df_original[col] = pd.to_numeric(df_original[col], errors='coerce')
df_original['NOTA_R'] = pd.to_numeric(df_original['NOTA_R'], errors='coerce')

df_profile = df_original.loc[df_train_sample.index, cols_notas].copy()
df_profile['NOME_CURSO'] = df_train_sample['NOME_CURSO'].values
df_profile['AREA_CONHECIMENTO'] = df_train_sample['AREA_CONHECIMENTO'].values
df_profile['CLUSTER'] = df_train_sample['CLUSTER'].values

print(f"\n{'='*20} TABELA DE MÉDIAS (NOTAS ORIGINAIS) POR CLUSTER {'='*20}")
tabela_medias = df_profile.groupby('CLUSTER')[cols_notas].mean().round(2)
print(tabela_medias)
tabela_medias.to_csv(os.path.join('data', 'case01', 'perfil_medias_clusters.csv'))

print(f"\n{'='*20} DISTRIBUIÇÃO POR ÁREA DO CONHECIMENTO (%) {'='*20}")
cruzamento_areas = pd.crosstab(
    df_profile['CLUSTER'], 
    df_profile['AREA_CONHECIMENTO'], 
    normalize='index'
) * 100
print(cruzamento_areas.round(2).astype(str) + '%')
cruzamento_areas.to_csv(os.path.join('data', 'case01', 'perfil_cruzamento_areas.csv'))

print(f"\n{'='*20} DETALHAMENTO DE PERSONAS E TOP CURSOS {'='*20}")
for cluster_id in range(3):
    df_c = df_profile[df_profile['CLUSTER'] == cluster_id]
    top_cursos = df_c['NOME_CURSO'].value_counts().head(5)
    area_predominante = df_c['AREA_CONHECIMENTO'].mode()[0]
    
    print(f"\n--- CLUSTER {cluster_id} ---")
    print(f"Área Predominante Escolhida: {area_predominante}")
    print("Médias das Notas nas Matérias:")
    print(tabela_medias.loc[cluster_id])
    print("\nTop 5 Cursos Mais Procurados:")
    print(top_cursos)

## 6. Conclusões Gerais e Diagnóstico da Parte 1

1. **Comparativo dos Algoritmos:**
   - O **K-Means** mostrou-se extremamente rápido e eficiente para particionar dados de larga escala, identificando a inclinação do cotovelo em $K=3$.
   - O **DBSCAN** encontrou dificuldades devido à natureza contínua e moderadamente homogênea das notas do ENEM no espaço $\mathbb{R}^5$, aglutinando a maioria das amostras em um único cluster denso.
   - O **Agrupamento Hierárquico Aglomerativo (Ligação Complete)** obteve o maior índice de separabilidade estatística (Silhueta $S \approx 0.65$), mostrando-se o modelo mais adequado para a caracterização das personas no SiSU.
2. **Validação da Hipótese de Negócio/Domínio:**
   - O cruzamento das personas com a coluna `AREA_CONHECIMENTO` confirmou a hipótese inicial: candidatos com desempenho superior em Matemática/Natureza escolhem prioritariamente engenharias e cursos de tecnologia, enquanto candidatos com notas destacadas em Redação/Humanas direcionam-se majoritariamente para Direito e licenciaturas.